# UPF Literature — Author Co-authorship Network

Builds and analyses a co-authorship network from the edge list produced by
`upf_bibliometrics.py`.  
**Pre-requisite:** run `python upf_bibliometrics.py` (or `--dry-run` for a
quick test) so that `output/coauthorship_edges.csv` exists.

**Install once** (if not already present):
```bash
pip install networkx plotly
```

**To export the interactive HTML dashboard**, run all cells then open
`output/upf_dashboard.html` in any browser — no Python required.
(`nbconvert --to html` renders the notebook with code cells; the dashboard
cell at the bottom produces the clean standalone file instead.)

In [ ]:
import collections
import math
import os
import warnings

import numpy as np

import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import networkx as nx
import pandas as pd
import plotly
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

# Use CDN-backed renderer so nbconvert can embed charts as HTML
pio.renderers.default = 'notebook_connected'

warnings.filterwarnings('ignore')

# ── Paths (relative to notebook location) ─────────────────────────────────────
EDGES_CSV        = os.path.join('..', 'output', 'coauthorship_edges.csv')
AUTHORS_CSV      = os.path.join('..', 'output', 'papers_by_author.csv')
INSTITUTIONS_CSV    = os.path.join('..', 'output', 'papers_by_institution.csv')
FUNDERS_CSV         = os.path.join('..', 'output', 'papers_by_funder.csv')
DEPT_CSV            = os.path.join('..', 'output', 'papers_by_department.csv')
FUNDING_COUNTRY_CSV = os.path.join('..', 'output', 'funding_by_country.csv')
YEAR_CSV            = os.path.join('..', 'output', 'papers_by_year.csv')
COUNTRY_YEAR_CSV    = os.path.join('..', 'output', 'papers_by_country_year.csv')
NET_METRICS_CSV     = os.path.join('..', 'output', 'network_metrics_by_year.csv')
EDGES_YEAR_CSV      = os.path.join('..', 'output', 'coauthorship_edges_by_year.csv')
OUTPUT_DIR          = os.path.join('..', 'output')

# ── Tunable parameters ─────────────────────────────────────────────────────────
MIN_PAPERS      = 3    # keep only authors with >= this many papers in the corpus
MIN_EDGE_WEIGHT = 2    # keep only co-authorship pairs that share >= this many papers
TOP_N_LABELS    = 30   # label the top-N highest-degree nodes in the graph plot
TOP_N_RANKING   = 25   # entries shown in the institution / author ranking charts
LAYOUT_SEED     = 42

print('NetworkX version:', nx.__version__)
print('Plotly version  :', plotly.__version__)

## 0. Quick Search

Set `SEARCH_AUTHOR` or `SEARCH_INSTITUTION` to a substring (case-insensitive)
and run this cell to look up any name before diving into the full analysis.

In [ ]:
SEARCH_AUTHOR      = ""   # e.g. "Monteiro"  or  "Touvier"
SEARCH_INSTITUTION = ""   # e.g. "São Paulo"  or  "Deakin"

def _search(df, col, query, extra_cols):
    if not query.strip():
        return None
    mask = df[col].str.contains(query, case=False, na=False)
    result = df[mask].sort_values('papers', ascending=False)
    return result[extra_cols]

# Authors
if SEARCH_AUTHOR:
    res = _search(
        authors_df, 'author_name', SEARCH_AUTHOR,
        ['author_name', 'institution', 'country', 'papers', 'citations']
    )
    print(f"Authors matching '{SEARCH_AUTHOR}':")
    display(res.head(20)) if res is not None and len(res) else print("  (no matches)")

# Institutions
if SEARCH_INSTITUTION:
    res = _search(
        institutions_df, 'institution', SEARCH_INSTITUTION,
        ['institution', 'country', 'papers', 'citations']
    )
    print(f"Institutions matching '{SEARCH_INSTITUTION}':")
    display(res.head(20)) if res is not None and len(res) else print("  (no matches)")

if not SEARCH_AUTHOR and not SEARCH_INSTITUTION:
    print("Set SEARCH_AUTHOR or SEARCH_INSTITUTION above and re-run.")

## 1. Load data

In [ ]:
edges_df        = pd.read_csv(EDGES_CSV)
authors_df      = pd.read_csv(AUTHORS_CSV)
institutions_df = pd.read_csv(INSTITUTIONS_CSV)
try:
    funders_df     = pd.read_csv(FUNDERS_CSV)
    funding_cty_df = pd.read_csv(FUNDING_COUNTRY_CSV)
    dept_df        = pd.read_csv(DEPT_CSV)
except FileNotFoundError:
    funders_df = funding_cty_df = pd.DataFrame()
    dept_df = pd.DataFrame()
    print('⚠ Funding/department CSVs not found — re-run upf_bibliometrics.py first')

# Drop the "no institution" catch-all row
institutions_df = institutions_df[
    institutions_df['institution'].notna() & (institutions_df['institution'] != '')
].copy()

print(f'Edge rows         : {len(edges_df):,}')
print(f'Author rows       : {len(authors_df):,}')
print(f'Institution rows  : {len(institutions_df):,}')
edges_df.head(3)

## 2. Build the full graph, then filter

We keep only nodes (authors) that appear in at least `MIN_PAPERS` papers **and**
edges (co-authorship pairs) with at least `MIN_EDGE_WEIGHT` shared papers.
This removes noise from one-off collaborations and focuses the network on
the productive core of the field.

In [ ]:
# Node attribute lookup: author_id → {name, institution, country, papers}
# groupby+first handles any duplicate author_id rows (e.g. authors with no
# OpenAlex ID whose fallback key collides across works)
node_attrs = (
    authors_df
    .groupby('author_id', as_index=True)
    .first()[['author_name', 'institution', 'country', 'papers', 'citations']]
    .to_dict(orient='index')
)

# Productive-author set
core_authors = {
    aid for aid, attr in node_attrs.items()
    if attr['papers'] >= MIN_PAPERS
}
print(f'Authors with >= {MIN_PAPERS} papers : {len(core_authors):,}')

# Build graph
G = nx.Graph()

for _, row in edges_df.iterrows():
    a1, a2, w = row['author1_id'], row['author2_id'], row['shared_papers']
    if a1 not in core_authors or a2 not in core_authors:
        continue
    if w < MIN_EDGE_WEIGHT:
        continue
    if G.has_edge(a1, a2):
        G[a1][a2]['weight'] += w
    else:
        G.add_edge(a1, a2, weight=w)

# Attach node attributes
for node in G.nodes():
    attrs = node_attrs.get(node, {})
    G.nodes[node]['name']        = attrs.get('author_name', node)
    G.nodes[node]['institution'] = attrs.get('institution', '')
    G.nodes[node]['country']     = attrs.get('country', '')
    G.nodes[node]['papers']      = attrs.get('papers', 0)
    G.nodes[node]['citations']    = attrs.get('citations', 0)

# Focus on the largest connected component
lcc_nodes = max(nx.connected_components(G), key=len)
G_lcc     = G.subgraph(lcc_nodes).copy()

print(f'Full graph        : {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges')
print(f'Largest component : {G_lcc.number_of_nodes():,} nodes, {G_lcc.number_of_edges():,} edges')


## 3. Global network statistics

In [4]:
n = G_lcc.number_of_nodes()
m = G_lcc.number_of_edges()
density   = nx.density(G_lcc)
avg_deg   = 2 * m / n if n else 0
avg_clust = nx.average_clustering(G_lcc, weight='weight')
diameter  = nx.diameter(G_lcc) if n < 5_000 else 'skipped (graph too large)'
avg_path  = nx.average_shortest_path_length(G_lcc) if n < 5_000 else 'skipped'
components_full = nx.number_connected_components(G)

print('── Network statistics (largest component) ──────────────')
print(f'  Nodes                 : {n:,}')
print(f'  Edges                 : {m:,}')
print(f'  Density               : {density:.4f}')
print(f'  Average degree        : {avg_deg:.2f}')
print(f'  Average clustering    : {avg_clust:.4f}')
print(f'  Diameter              : {diameter}')
print(f'  Avg shortest path     : {avg_path}')
print(f'  Components (full graph): {components_full:,}')

── Network statistics (largest component) ──────────────
  Nodes                 : 1,380
  Edges                 : 8,921
  Density               : 0.0094
  Average degree        : 12.93
  Average clustering    : 0.0397
  Diameter              : 13
  Avg shortest path     : 4.446804552763502
  Components (full graph): 172


## 4. Centrality measures

In [ ]:
degree_cent     = nx.degree_centrality(G_lcc)
betweenness     = nx.betweenness_centrality(G_lcc, weight='weight', normalized=True)
pagerank        = nx.pagerank(G_lcc, weight='weight')
clustering      = nx.clustering(G_lcc, weight='weight')

centrality_df = pd.DataFrame({
    'author_id'   : list(G_lcc.nodes()),
    'name'        : [G_lcc.nodes[n]['name']        for n in G_lcc.nodes()],
    'institution' : [G_lcc.nodes[n]['institution'] for n in G_lcc.nodes()],
    'country'     : [G_lcc.nodes[n]['country']     for n in G_lcc.nodes()],
    'papers'      : [G_lcc.nodes[n]['papers']      for n in G_lcc.nodes()],
    'citations'   : [G_lcc.nodes[n].get('citations', 0) for n in G_lcc.nodes()],
    'degree'      : [G_lcc.degree(n)               for n in G_lcc.nodes()],
    'degree_centrality'  : [degree_cent[n]   for n in G_lcc.nodes()],
    'betweenness'        : [betweenness[n]   for n in G_lcc.nodes()],
    'pagerank'           : [pagerank[n]      for n in G_lcc.nodes()],
    'clustering'         : [clustering[n]   for n in G_lcc.nodes()],
})

centrality_df.sort_values('betweenness', ascending=False, inplace=True)
centrality_df.reset_index(drop=True, inplace=True)

# Save
out_path = os.path.join(OUTPUT_DIR, 'author_centrality.csv')
centrality_df.to_csv(out_path, index=False)
print(f'Saved → {out_path}')

print('\nTop 15 by betweenness centrality (bridges / gatekeepers):')
centrality_df[['name', 'institution', 'country', 'papers', 'degree', 'betweenness', 'pagerank']].head(15)

In [6]:
print('Top 15 by degree (most direct collaborators):')
centrality_df.sort_values('degree', ascending=False).head(15)[
    ['name', 'institution', 'country', 'papers', 'degree', 'betweenness']
]

Top 15 by degree (most direct collaborators):


,name,institution,country,papers,degree,betweenness
2,Renata Bertazzi Levy,Universidade de São Paulo,BR,143,165,0.107005
1,Carlos Augusto Monteiro,Universidade de São Paulo,BR,156,139,0.129257
4,Fernanda Rauber,Universidade de São Paulo,BR,93,138,0.098458
5,Eurídice Martínez Steele,Universidade de São Paulo,BR,133,120,0.096528
0,Neha Khandpur,Universidade de São Paulo,BR,96,116,0.142175
16,Christopher Millett,Universidade de São Paulo,BR,45,115,0.043840
40,Mathilde Touvier,Sorbonne Paris Cité,FR,71,105,0.022435
29,Bernard Srour,Equipe de Recherche en Epidémiologie Nutrition...,FR,61,103,0.029632
50,Inge Huybrechts,Centre international de recherche sur le cancer,FR,33,86,0.018655
91,Eszter P. Vamos,Imperial College London,GB,35,81,0.008897


## 5. Top-N Rankings — Institutions and Authors

Interactive bar charts. Hover over any bar for full details.  
Change `TOP_N_RANKING` in the parameters cell (cell 1) to show more or fewer entries.

In [7]:
# ── Top-N Institutions ────────────────────────────────────────────────────────
inst_by_papers = (
    institutions_df.nlargest(TOP_N_RANKING, 'papers')
    .assign(label=lambda df: df['institution'].str[:55])
    .sort_values('papers')          # ascending → largest at top in horizontal bar
)
inst_by_cites = (
    institutions_df.nlargest(TOP_N_RANKING, 'citations')
    .assign(label=lambda df: df['institution'].str[:55])
    .sort_values('citations')
)

fig_inst = go.Figure()

fig_inst.add_trace(go.Bar(
    x=inst_by_papers['papers'],
    y=inst_by_papers['label'],
    orientation='h',
    name='Papers',
    visible=True,
    customdata=inst_by_papers[['institution', 'country', 'citations']].values,
    hovertemplate=(
        '<b>%{customdata[0]}</b><br>'
        'Country: %{customdata[1]}<br>'
        'Papers: %{x:,}<br>'
        'Citations: %{customdata[2]:,}<extra></extra>'
    ),
    marker_color='steelblue',
))

fig_inst.add_trace(go.Bar(
    x=inst_by_cites['citations'],
    y=inst_by_cites['label'],
    orientation='h',
    name='Citations',
    visible=False,
    customdata=inst_by_cites[['institution', 'country', 'papers']].values,
    hovertemplate=(
        '<b>%{customdata[0]}</b><br>'
        'Country: %{customdata[1]}<br>'
        'Citations: %{x:,}<br>'
        'Papers: %{customdata[2]:,}<extra></extra>'
    ),
    marker_color='coral',
))

fig_inst.update_layout(
    title=f'Top {TOP_N_RANKING} Institutions by Paper Count',
    height=700,
    xaxis_title='Papers',
    yaxis_title='',
    showlegend=False,
    margin=dict(l=10, r=20, t=60, b=40),
    updatemenus=[dict(
        type='buttons',
        direction='right',
        x=0.0, xanchor='left',
        y=1.08, yanchor='top',
        buttons=[
            dict(
                label='By Papers',
                method='update',
                args=[{'visible': [True, False]},
                      {'title': f'Top {TOP_N_RANKING} Institutions by Paper Count',
                       'xaxis.title.text': 'Papers'}],
            ),
            dict(
                label='By Citations',
                method='update',
                args=[{'visible': [False, True]},
                      {'title': f'Top {TOP_N_RANKING} Institutions by Citation Count',
                       'xaxis.title.text': 'Citations'}],
            ),
        ],
    )],
)
fig_inst.show()

In [8]:
# ── Top-N Authors ─────────────────────────────────────────────────────────────
authors_clean = authors_df[
    authors_df['author_name'].notna() & (authors_df['author_name'] != '')
].copy()

auth_by_papers = (
    authors_clean.nlargest(TOP_N_RANKING, 'papers')
    .sort_values('papers')
)
auth_by_cites = (
    authors_clean.nlargest(TOP_N_RANKING, 'citations')
    .sort_values('citations')
)

fig_auth = go.Figure()

fig_auth.add_trace(go.Bar(
    x=auth_by_papers['papers'],
    y=auth_by_papers['author_name'],
    orientation='h',
    name='Papers',
    visible=True,
    customdata=auth_by_papers[['institution', 'country', 'citations']].values,
    hovertemplate=(
        '<b>%{y}</b><br>'
        'Institution: %{customdata[0]}<br>'
        'Country: %{customdata[1]}<br>'
        'Papers: %{x:,}<br>'
        'Citations: %{customdata[2]:,}<extra></extra>'
    ),
    marker_color='steelblue',
))

fig_auth.add_trace(go.Bar(
    x=auth_by_cites['citations'],
    y=auth_by_cites['author_name'],
    orientation='h',
    name='Citations',
    visible=False,
    customdata=auth_by_cites[['institution', 'country', 'papers']].values,
    hovertemplate=(
        '<b>%{y}</b><br>'
        'Institution: %{customdata[0]}<br>'
        'Country: %{customdata[1]}<br>'
        'Citations: %{x:,}<br>'
        'Papers: %{customdata[2]:,}<extra></extra>'
    ),
    marker_color='coral',
))

fig_auth.update_layout(
    title=f'Top {TOP_N_RANKING} Authors by Paper Count',
    height=700,
    xaxis_title='Papers',
    yaxis_title='',
    showlegend=False,
    margin=dict(l=10, r=20, t=60, b=40),
    updatemenus=[dict(
        type='buttons',
        direction='right',
        x=0.0, xanchor='left',
        y=1.08, yanchor='top',
        buttons=[
            dict(
                label='By Papers',
                method='update',
                args=[{'visible': [True, False]},
                      {'title': f'Top {TOP_N_RANKING} Authors by Paper Count',
                       'xaxis.title.text': 'Papers'}],
            ),
            dict(
                label='By Citations',
                method='update',
                args=[{'visible': [False, True]},
                      {'title': f'Top {TOP_N_RANKING} Authors by Citation Count',
                       'xaxis.title.text': 'Citations'}],
            ),
        ],
    )],
)
fig_auth.show()

## 5. Community detection

Uses the Louvain algorithm (built into NetworkX ≥ 2.7) to partition the
network into research communities.

In [ ]:
communities = nx.community.louvain_communities(G_lcc, weight='weight', seed=LAYOUT_SEED)
communities = sorted(communities, key=len, reverse=True)

print(f'Number of communities detected: {len(communities)}')
print(f'Sizes of top-10 communities   : {[len(c) for c in communities[:10]]}')

# Assign community labels to nodes
node_community = {}
for idx, community in enumerate(communities):
    for node in community:
        node_community[node] = idx

nx.set_node_attributes(G_lcc, node_community, 'community')
centrality_df['community'] = centrality_df['author_id'].map(node_community)

# Build per-community summary
comm_summary = (
    centrality_df
    .groupby('community')
    .agg(
        size=('name', 'count'),
        total_papers=('papers', 'sum'),
        total_citations=('citations', 'sum'),
        top_country=('country', lambda x: x.value_counts().index[0] if len(x) else ''),
        top_institution=('institution', lambda x: x.value_counts().index[0] if len(x) else ''),
        # Top author by betweenness within the community
        top_author=('name', lambda x: (
            centrality_df.loc[x.index]
            .sort_values('betweenness', ascending=False)['name'].iloc[0]
            if len(x) else ''
        )),
    )
    .reset_index()
    .sort_values('size', ascending=False)
)

# Build human-readable community labels: "Country · Lastname (N)"
def _make_label(row):
    lastname = row['top_author'].split()[-1] if row['top_author'] else '?'
    country  = row['top_country'] or '?'
    n        = int(row['size'])
    return f"{country} · {lastname} ({n})"

community_names = {
    int(row['community']): _make_label(row)
    for _, row in comm_summary.iterrows()
}
nx.set_node_attributes(G_lcc, {n: community_names.get(v, str(v))
                                for n, v in node_community.items()}, 'community_label')
centrality_df['community_label'] = centrality_df['community'].map(community_names)

print('\nTop-10 communities:')
comm_summary[['community', 'size', 'total_papers', 'total_citations',
              'top_country', 'top_author']].head(10)

## 6. Visualisations

### 6a. Degree distribution

In [10]:
degrees = [d for _, d in G_lcc.degree()]
freq    = collections.Counter(degrees)
deg_df  = pd.DataFrame({'degree': list(freq.keys()), 'count': list(freq.values())}).sort_values('degree')

fig_deg = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Degree Distribution (linear)', 'Degree Distribution (log–log)'],
)

fig_deg.add_trace(
    go.Histogram(x=degrees, nbinsx=40, name='Count', marker_color='steelblue'),
    row=1, col=1,
)
fig_deg.add_trace(
    go.Scatter(
        x=deg_df['degree'], y=deg_df['count'],
        mode='markers',
        marker=dict(size=5, color='steelblue', opacity=0.7),
        hovertemplate='Degree: %{x}<br>Count: %{y}<extra></extra>',
        name='Count',
    ),
    row=1, col=2,
)

fig_deg.update_xaxes(title_text='Degree', row=1, col=1)
fig_deg.update_yaxes(title_text='Count', row=1, col=1)
fig_deg.update_xaxes(type='log', title_text='Degree (log)', row=1, col=2)
fig_deg.update_yaxes(type='log', title_text='Count (log)', row=1, col=2)
fig_deg.update_layout(
    height=420,
    showlegend=False,
    title_text=(
        f'Degree Distribution  |  '
        f'Mean: {sum(degrees)/len(degrees):.2f}  |  Max: {max(degrees)}'
    ),
)
fig_deg.show()
print(f'Mean degree: {sum(degrees)/len(degrees):.2f}  Max: {max(degrees)}')

Mean degree: 12.93  Max: 165


**Reading this chart:**  
The left panel shows how many authors have each number of co-authors (degree).
A long right tail — with a few authors having very many collaborators and most
having only a few — is typical of real-world collaboration networks and suggests
a **scale-free** structure.  
The log–log panel (right) linearises that tail: if the points follow a straight
line it is consistent with a power-law degree distribution, which characterises
highly unequal networks where a small hub dominates.  
**What to look for:** a very steep slope means collaboration is concentrated in
a few prolific connectors; a shallower slope indicates a more distributed network.

### 6b. Co-authorship network (largest component, coloured by community)

In [ ]:
# Cap for legibility
PLOT_CAP = 500
if G_lcc.number_of_nodes() > PLOT_CAP:
    top_nodes = sorted(G_lcc.nodes(), key=lambda n: G_lcc.degree(n), reverse=True)[:PLOT_CAP]
    G_plot = G_lcc.subgraph(top_nodes).copy()
    print(f'Plotting top-{PLOT_CAP} nodes by degree ({G_lcc.number_of_nodes():,} total)')
else:
    G_plot = G_lcc

pos = nx.spring_layout(G_plot, weight='weight', seed=LAYOUT_SEED, k=0.8)

# ── Build Plotly traces ───────────────────────────────────────────────────────
# Edges
edge_x, edge_y = [], []
for u, v in G_plot.edges():
    x0, y0 = pos[u]; x1, y1 = pos[v]
    edge_x += [x0, x1, None]; edge_y += [y0, y1, None]

edge_trace = go.Scatter(
    x=edge_x, y=edge_y, mode='lines',
    line=dict(width=0.5, color='#888'), hoverinfo='none',
    showlegend=False,
)

# Nodes — one trace per community for legend
comm_ids = sorted(set(G_plot.nodes[n].get('community', 0) for n in G_plot.nodes()))
palette  = px.colors.qualitative.Alphabet
node_traces = []

for cid in comm_ids:
    nodes_in_comm = [n for n in G_plot.nodes() if G_plot.nodes[n].get('community', 0) == cid]
    label = G_plot.nodes[nodes_in_comm[0]].get('community_label', str(cid)) if nodes_in_comm else str(cid)
    xs = [pos[n][0] for n in nodes_in_comm]
    ys = [pos[n][1] for n in nodes_in_comm]
    sizes  = [6 + 2 * G_plot.degree(n) for n in nodes_in_comm]
    texts  = [G_plot.nodes[n].get('name', n).split()[-1]
               if G_plot.degree(n) >= sorted([G_plot.degree(n) for n in G_plot.nodes()])[-TOP_N_LABELS]
               else '' for n in nodes_in_comm]
    hovers = [f"{G_plot.nodes[n].get('name','')}<br>{G_plot.nodes[n].get('institution','')}<br>"
               f"{G_plot.nodes[n].get('country','')}  deg={G_plot.degree(n)}"
               for n in nodes_in_comm]
    node_traces.append(go.Scatter(
        x=xs, y=ys, mode='markers+text',
        marker=dict(size=sizes, color=palette[cid % len(palette)], opacity=0.85,
                    line=dict(width=0.5, color='white')),
        text=texts, textposition='top center', textfont=dict(size=7),
        hovertext=hovers, hoverinfo='text',
        name=label, legendgroup=str(cid),
    ))

fig_network = go.Figure(data=[edge_trace] + node_traces)
fig_network.update_layout(
    title=f'UPF Co-authorship Network  |  {G_plot.number_of_nodes()} authors, '
          f'{G_plot.number_of_edges()} edges  |  colour = community',
    showlegend=True,
    hovermode='closest',
    height=750,
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    legend=dict(title='Community', itemsizing='constant', font=dict(size=9)),
    margin=dict(l=10, r=10, t=50, b=10),
)
fig_network.show()

**Reading this chart:**  
Each dot is an author; edges connect co-authors.  
**Colour** groups authors into research communities detected by the Louvain
algorithm — nodes of the same colour tend to publish together more than with
other groups.  
**Size** reflects degree (number of direct co-authors): large nodes are the most
collaborative hubs.  
**Layout** uses a force-directed spring algorithm: nodes that share many
collaborators are pulled close together, so spatial proximity approximates
research community.  
**What to look for:** tight clusters indicate nationally or institutionally
cohesive groups; bridges between clusters are authors who connect otherwise
separate research communities.

### 6c. Betweenness vs degree scatter — identifying bridges

In [ ]:
# community_label is already built in the community-detection cell
plot_df = centrality_df.copy()
# Fallback for any unmapped nodes
plot_df['community_label'] = plot_df['community_label'].fillna('Other')

fig_scatter = px.scatter(
    plot_df,
    x='degree',
    y='betweenness',
    size='papers',
    color='community_label',
    hover_name='name',
    hover_data={
        'institution': True,
        'country': True,
        'papers': True,
        'degree': True,
        'betweenness': ':.4f',
        'pagerank': ':.4f',
        'community_label': False,
    },
    title='Degree vs Betweenness Centrality  |  point size ∝ papers  |  colour = community',
    labels={
        'degree': 'Degree (number of co-authors)',
        'betweenness': 'Betweenness centrality',
        'community_label': 'Community',
    },
    color_discrete_sequence=px.colors.qualitative.Alphabet,
    size_max=30,
    height=650,
)
fig_scatter.update_traces(marker_opacity=0.7)
fig_scatter.update_layout(legend_title_text='Community')
fig_scatter.show()

**Reading this chart:**  
- **X axis (degree):** how many distinct co-authors an author has.  
  High degree = a well-connected hub who works across many collaborations.  
- **Y axis (betweenness centrality):** how often an author lies on the shortest
  path between two other authors.  
  High betweenness = a *broker* or *bridge* — someone who connects otherwise
  separate research groups, even if they do not have the most co-authors overall.  
- **Point size** is proportional to the number of papers published.  
- **Colour** identifies the Louvain community (label = country · top author).  

**Four quadrant interpretation:**  
| | Low betweenness | High betweenness |
|---|---|---|
| **High degree** | Core hub within one community | Cross-community connector (key bridge) |
| **Low degree** | Peripheral author | Structural hole broker |

Authors in the top-right are the most strategically important for knowledge
transfer across the field.

### 6d. Country-level collaboration heatmap

Counts co-authored papers between pairs of countries.

In [ ]:
def _safe_country(val):
    if val is None or (isinstance(val, float) and math.isnan(val)):
        return ''
    return str(val)

country_edges = collections.Counter()
for u, v, data in G_lcc.edges(data=True):
    c1 = _safe_country(G_lcc.nodes[u].get('country', ''))
    c2 = _safe_country(G_lcc.nodes[v].get('country', ''))
    if c1 and c2 and c1 != c2:
        pair = tuple(sorted([c1, c2]))
        country_edges[pair] += data.get('weight', 1)

top_pairs = country_edges.most_common(20)
countries_involved = sorted({c for pair, _ in top_pairs for c in pair})

matrix = pd.DataFrame(0, index=countries_involved, columns=countries_involved)
for (c1, c2), w in top_pairs:
    matrix.loc[c1, c2] = w
    matrix.loc[c2, c1] = w

fig_heatmap = go.Figure(go.Heatmap(
    z=matrix.values.tolist(),
    x=list(matrix.columns),
    y=list(matrix.index),
    colorscale='YlOrRd',
    hoverongaps=False,
    hovertemplate='%{y} ↔ %{x}<br>Co-authored papers: %{z}<extra></extra>',
    colorbar=dict(title='Papers'),
))
fig_heatmap.update_layout(
    title='Cross-country Co-authorship (top-20 pairs)',
    height=520,
    xaxis=dict(tickangle=45),
    margin=dict(l=60, r=20, t=60, b=80),
)
fig_heatmap.show()

## 8. Citation Impact Analysis

The co-authorship network above captures *collaboration* structure.
This section uses the `citations` column to explore *impact* structure:
which authors, communities and countries generate the most-cited work,
and how efficiently (citations per paper).

> **Note on a full citation network:** A directed paper→paper citation graph
> would require fetching `referenced_works` for each paper from OpenAlex —
> an additional API pass of comparable scale to the main fetch.
> The analysis below uses the aggregate citation counts already retrieved.

In [ ]:
# ── 8a. Papers vs Citations scatter (author level) ────────────────────────────
plot_cit = centrality_df[centrality_df['papers'] >= MIN_PAPERS].copy()
plot_cit['cit_per_paper'] = (plot_cit['citations'] / plot_cit['papers']).round(1)
plot_cit['community_label'] = plot_cit['community_label'].fillna('Other')

fig_cit = px.scatter(
    plot_cit,
    x='papers',
    y='citations',
    size='degree',
    color='community_label',
    hover_name='name',
    hover_data={
        'institution': True,
        'country': True,
        'cit_per_paper': True,
        'degree': True,
        'community_label': False,
    },
    log_x=True,
    log_y=True,
    title='Author Citation Impact  |  size ∝ degree  |  colour = community<br>'
          '<sup>Log scales — top-right = many papers AND highly cited</sup>',
    labels={
        'papers': 'Papers (log)',
        'citations': 'Total citations (log)',
        'community_label': 'Community',
    },
    color_discrete_sequence=px.colors.qualitative.Alphabet,
    size_max=25,
    height=600,
)
fig_cit.update_traces(marker_opacity=0.7)
fig_cit.show()

# ── 8b. Citation efficiency by community ──────────────────────────────────────
comm_cit = (
    centrality_df
    .groupby('community_label')
    .agg(
        authors=('name', 'count'),
        total_papers=('papers', 'sum'),
        total_citations=('citations', 'sum'),
    )
    .assign(cit_per_paper=lambda df: (df['total_citations'] / df['total_papers']).round(1))
    .sort_values('total_citations', ascending=False)
    .head(15)
    .reset_index()
)

fig_comm_cit = go.Figure()
fig_comm_cit.add_trace(go.Bar(
    x=comm_cit['total_citations'],
    y=comm_cit['community_label'],
    orientation='h',
    name='Total citations',
    marker_color='steelblue',
    customdata=comm_cit[['total_papers', 'cit_per_paper', 'authors']].values,
    hovertemplate='<b>%{y}</b><br>Citations: %{x:,}<br>Papers: %{customdata[0]:,}'
                  '<br>Cit/paper: %{customdata[1]}<br>Authors: %{customdata[2]}<extra></extra>',
))
fig_comm_cit.update_layout(
    title='Total Citations by Research Community (top 15)',
    xaxis_title='Total citations',
    height=500,
    yaxis=dict(autorange='reversed'),
)
fig_comm_cit.show()

# ── 8c. Citations per paper by country ────────────────────────────────────────
country_df = pd.read_csv(os.path.join(OUTPUT_DIR, '..', 'output', 'papers_by_country.csv'))
country_df = country_df[country_df['country'].notna() & (country_df['country'] != '')].copy()
country_df['cit_per_paper'] = (country_df['citations'] / country_df['papers']).round(1)
top_countries = country_df.nlargest(20, 'papers')

fig_cty = make_subplots(rows=1, cols=2,
    subplot_titles=['Papers by Country (top 20)', 'Citations per Paper (top 20 by papers)'])

fig_cty.add_trace(go.Bar(
    x=top_countries['papers'], y=top_countries['country'],
    orientation='h', name='Papers', marker_color='steelblue',
    hovertemplate='<b>%{y}</b>: %{x:,} papers<extra></extra>',
), row=1, col=1)

fig_cty.add_trace(go.Bar(
    x=top_countries.sort_values('cit_per_paper', ascending=True)['cit_per_paper'],
    y=top_countries.sort_values('cit_per_paper', ascending=True)['country'],
    orientation='h', name='Cit/paper', marker_color='coral',
    hovertemplate='<b>%{y}</b>: %{x} cit/paper<extra></extra>',
), row=1, col=2)

fig_cty.update_layout(height=550, showlegend=False,
    title_text='Country-level Output vs Impact')
fig_cty.update_yaxes(autorange='reversed', row=1, col=1)
fig_cty.show()
print('\n**Interpretation:** Volume (papers) and impact (citations per paper) often diverge.\n'
      'Countries with fewer but more-cited papers punch above their weight in influence.')

**Reading these charts:**

**8a — Author citation impact (scatter):**  
Both axes are log-scaled. Authors in the **top-right** are highly prolific *and*
highly cited — the field's most influential researchers.  
Authors in the **top-left** have few papers but each is highly cited (early-career
or review-paper specialists); those in the **bottom-right** are prolific but
lower-impact.

**8b — Citations by community:**  
Shows which research cluster generates the most total citation impact.  
Compare `total_citations` with `total_papers` to see whether a community's
influence is driven by volume or quality.

**8c — Country output vs efficiency:**  
The left panel ranks by paper count (volume); the right panel shows citations
per paper (efficiency/quality).  
High volume + low efficiency may indicate a large but less internationally
visible community; low volume + high efficiency suggests a small but punchy
research base.

## 9. Funding Analysis

Which funders and countries drive the UPF literature?  
OpenAlex records the `grants` field where available — coverage is partial
(not all publishers deposit funding metadata), so interpret percentages as
lower bounds.

In [ ]:
# ── 9a. Top funders by paper count ───────────────────────────────────────────
top_funders = funders_df[funders_df['funder_name'].notna()].head(25).copy()
top_funders['cit_per_paper'] = (top_funders['citations'] / top_funders['papers']).round(1)
top_funders_plot = top_funders.sort_values('papers')

fig_funders = go.Figure()
fig_funders.add_trace(go.Bar(
    x=top_funders_plot['papers'],
    y=top_funders_plot['funder_name'],
    orientation='h',
    name='Papers',
    marker_color='steelblue',
    customdata=top_funders_plot[['citations', 'cit_per_paper']].values,
    hovertemplate='<b>%{y}</b><br>Papers: %{x:,}<br>'
                  'Citations: %{customdata[0]:,}<br>'
                  'Cit/paper: %{customdata[1]}<extra></extra>',
))
fig_funders.update_layout(
    title='Top 25 Funders by Funded Paper Count',
    xaxis_title='Papers',
    height=600,
    yaxis=dict(autorange='reversed'),
    margin=dict(l=300),
)
fig_funders.show()

# ── 9b. Funding rate by country ───────────────────────────────────────────────
top_cty = funding_cty_df.nlargest(20, 'papers').sort_values('pct_funded')

fig_fund_cty = make_subplots(
    rows=1, cols=2,
    subplot_titles=['% Papers with Funding Acknowledged',
                    'Funded vs Unfunded Papers'],
)
fig_fund_cty.add_trace(go.Bar(
    x=top_cty['pct_funded'], y=top_cty['country'],
    orientation='h', name='% Funded', marker_color='teal',
    hovertemplate='<b>%{y}</b>: %{x}% funded<extra></extra>',
), row=1, col=1)

top_cty_s = top_cty.sort_values('papers')
fig_fund_cty.add_trace(go.Bar(
    x=top_cty_s['funded_papers'], y=top_cty_s['country'],
    orientation='h', name='Funded', marker_color='teal', opacity=0.8,
), row=1, col=2)
fig_fund_cty.add_trace(go.Bar(
    x=top_cty_s['papers'] - top_cty_s['funded_papers'],
    y=top_cty_s['country'],
    orientation='h', name='No funding record', marker_color='lightgrey',
), row=1, col=2)

fig_fund_cty.update_layout(
    barmode='stack', height=550,
    title_text='Funding Coverage by Country (top 20 by paper count)',
)
fig_fund_cty.update_yaxes(autorange='reversed', row=1, col=1)
fig_fund_cty.show()

# ── 9c. Summary table ─────────────────────────────────────────────────────────
total_funded = funders_df['papers'].sum()
print(f'Papers with at least one funder recorded: {total_funded:,}')
print()
print('Top 15 funders:')
display(top_funders[['funder_name','papers','citations','cit_per_paper']].head(15).reset_index(drop=True))

**Reading these charts:**

**9a — Top funders:**  
Paper count per funder (hover for citation impact).  
Funders with high citations-per-paper relative to their paper count are backing
high-impact research.  A paper with multiple funders is counted once per funder.

**9b — Funding coverage by country:**  
The left panel shows what percentage of a country's papers acknowledge external
funding — this reflects both actual funding rates and publisher metadata deposit
practices.  
The right panel stacks funded vs no-record papers; a large grey bar can mean
genuine self-funding *or* incomplete metadata.  
Countries with high % funded and high citations-per-paper (§8c) are particularly
well-supported research environments.

## 10. Departments *(experimental)*

Department names are extracted heuristically from authors' raw affiliation
strings as submitted to the publisher. Coverage is **partial** — only papers
where a recognisable department keyword was found are included. Treat these
rankings as indicative rather than definitive.

In [ ]:
if dept_df.empty:
    print('⚠ Department data not available — re-run upf_bibliometrics.py first')
else:
    top_dept = dept_df[dept_df['department'].notna()].head(25).copy()
    top_dept['label'] = top_dept.apply(
        lambda r: f"{r['department']} ({r['institution'][:30]})", axis=1
    )
    top_dept['cit_per_paper'] = (top_dept['citations'] / top_dept['papers']).round(1)
    top_dept_plot = top_dept.sort_values('papers')

    fig_dept = go.Figure()
    fig_dept.add_trace(go.Bar(
        x=top_dept_plot['papers'],
        y=top_dept_plot['label'],
        orientation='h',
        marker_color='mediumpurple',
        customdata=top_dept_plot[['institution','country','citations','cit_per_paper']].values,
        hovertemplate='<b>%{y}</b><br>Institution: %{customdata[0]}<br>'
                      'Country: %{customdata[1]}<br>Papers: %{x}<br>'
                      'Citations: %{customdata[2]:,}<br>Cit/paper: %{customdata[3]}<extra></extra>',
    ))
    fig_dept.update_layout(
        title='Top 25 Departments by Paper Count ⚠ Experimental — partial coverage',
        xaxis_title='Papers',
        height=650,
        yaxis=dict(autorange='reversed'),
        margin=dict(l=350),
    )
    fig_dept.show()
    print(f'Departments with data: {len(dept_df):,}  |  '
          f'Papers covered: {dept_df["papers"].sum():,}')

**Reading this chart:**  
Each bar shows the number of UPF papers attributed to a department within its
parent institution. Department names are taken directly from author affiliation
strings — the same text that appears on the published paper — so the same
department may appear under slightly different names across papers.  
Hover for institution, country, citation count, and citations per paper.  
**Coverage warning:** only papers where a department-like keyword was detected
in the affiliation string are included. Papers from journals that do not require
department-level affiliations will not appear here.

## §11 Temporal Analysis

How has UPF research grown over time and which countries are driving it?

In [ ]:
year_df         = pd.read_csv(YEAR_CSV)
country_year_df = pd.read_csv(COUNTRY_YEAR_CSV)
net_metrics_df  = pd.read_csv(NET_METRICS_CSV)

# Clip to 2000+ to exclude sparse early records
year_df         = year_df[year_df['year'] >= 2000].copy()
country_year_df = country_year_df[country_year_df['year'] >= 2000].copy()
net_metrics_df  = net_metrics_df[net_metrics_df['year'] >= 2000].copy()
print(f'Year range: {year_df["year"].min()} – {year_df["year"].max()}')

In [ ]:
# ── §11a. Annual publication count ──────────────────────────────────────
x = year_df['year'].values
y = year_df['papers'].values
z = np.polyfit(x, y, 1)
trend = np.poly1d(z)(x)

fig_year = go.Figure()
fig_year.add_trace(go.Bar(
    x=x, y=y, name='Papers published',
    marker_color='steelblue', opacity=0.85
))
fig_year.add_trace(go.Scatter(
    x=x, y=trend, mode='lines', name='Linear trend',
    line=dict(color='crimson', width=2, dash='dash')
))
fig_year.update_layout(
    title='UPF Publications per Year',
    xaxis_title='Year', yaxis_title='Papers published',
    template='plotly_white', bargap=0.25,
    legend=dict(x=0.02, y=0.98)
)
fig_year.show()

In [ ]:
# ── §11b. Top-8 countries over time ─────────────────────────────────────
TOP_N_COUNTRIES = 8
top_c = (
    country_year_df.groupby('country')['papers'].sum()
    .sort_values(ascending=False)
    .head(TOP_N_COUNTRIES).index.tolist()
)
pivot_ct = (
    country_year_df[country_year_df['country'].isin(top_c)]
    .pivot_table(index='year', columns='country', values='papers', fill_value=0)
    .reset_index()
)

fig_country_trend = go.Figure()
for c in top_c:
    if c in pivot_ct.columns:
        fig_country_trend.add_trace(go.Scatter(
            x=pivot_ct['year'], y=pivot_ct[c],
            mode='lines+markers', name=c, line=dict(width=2)
        ))
fig_country_trend.update_layout(
    title='Annual UPF Publications — Top 8 Countries',
    xaxis_title='Year', yaxis_title='Papers published',
    template='plotly_white',
    legend=dict(title='Country', x=1.01, y=0.99)
)
fig_country_trend.show()

In [ ]:
# ── §11c. Cumulative network growth ─────────────────────────────────────
fig_network_growth = go.Figure()
fig_network_growth.add_trace(go.Scatter(
    x=net_metrics_df['year'],
    y=net_metrics_df['cumulative_authors'],
    mode='lines+markers', name='Authors (nodes)',
    line=dict(color='steelblue', width=2)
))
fig_network_growth.add_trace(go.Scatter(
    x=net_metrics_df['year'],
    y=net_metrics_df['cumulative_edges'],
    mode='lines+markers', name='Collaborations (edges)',
    line=dict(color='tomato', width=2, dash='dot'),
    yaxis='y2'
))
fig_network_growth.update_layout(
    title='Cumulative Growth of the UPF Co-authorship Network',
    xaxis_title='Year',
    yaxis=dict(title='Cumulative authors', side='left', showgrid=True),
    yaxis2=dict(title='Cumulative collaborations', side='right', overlaying='y', showgrid=False),
    template='plotly_white',
    legend=dict(x=0.02, y=0.98)
)
fig_network_growth.show()

In [ ]:
# ── §11d. Animated co-authorship network (cumulative by year) ────────────────
try:
    edges_yr_df = pd.read_csv(EDGES_YEAR_CSV)
except FileNotFoundError:
    print('\u26a0 Re-run upf_bibliometrics.py to generate edges_by_year data')
    edges_yr_df = pd.DataFrame()

if not edges_yr_df.empty:
    known = set(pos.keys())
    ey = edges_yr_df[
        edges_yr_df['author1_id'].isin(known) &
        edges_yr_df['author2_id'].isin(known) &
        (edges_yr_df['year'] >= 2005)
    ].copy()

    anim_years = sorted(ey['year'].unique())

    # Node colour from community assignment (same palette as static network)
    node_color_map = {
        n: palette[G_plot.nodes[n].get('community', 0) % len(palette)]
        for n in G_plot.nodes()
    }

    pos_df = pd.DataFrame.from_dict(pos, orient='index', columns=['px', 'py'])

    def _anim_traces(cum_df):
        m = (
            cum_df
            .merge(pos_df.rename(columns={'px': 'x1', 'py': 'y1'}),
                   left_on='author1_id', right_index=True, how='inner')
            .merge(pos_df.rename(columns={'px': 'x2', 'py': 'y2'}),
                   left_on='author2_id', right_index=True, how='inner')
        )
        nans = np.full(len(m), np.nan)
        ex = np.stack([m['x1'].values, m['x2'].values, nans], axis=1).flatten()
        ey_c = np.stack([m['y1'].values, m['y2'].values, nans], axis=1).flatten()

        active = list(set(m['author1_id'].tolist()) | set(m['author2_id'].tolist()))
        nx_c   = [pos[n][0] for n in active]
        ny_c   = [pos[n][1] for n in active]
        nc     = [node_color_map.get(n, '#aaaaaa') for n in active]
        sizes  = [max(4, min(14, 4 + G_plot.degree(n))) if n in G_plot else 4 for n in active]
        hover  = [
            f"{G_lcc.nodes[n].get('name','')} | "
            f"{G_lcc.nodes[n].get('institution','')} | "
            f"{G_lcc.nodes[n].get('country','')}"
            if n in G_lcc else n for n in active
        ]
        return [
            go.Scatter(x=ex.tolist(), y=ey_c.tolist(), mode='lines',
                       line=dict(width=0.4, color='rgba(140,140,140,0.2)'),
                       hoverinfo='none', showlegend=False),
            go.Scatter(x=nx_c, y=ny_c, mode='markers',
                       marker=dict(size=sizes, color=nc,
                                   line=dict(width=0.5, color='white')),
                       hovertext=hover, hoverinfo='text', showlegend=False),
        ]

    frames = [
        go.Frame(data=_anim_traces(ey[ey['year'] <= y]), name=str(y))
        for y in anim_years
    ]
    init_traces = _anim_traces(ey[ey['year'] <= anim_years[0]])

    fig_anim_network = go.Figure(
        data=init_traces,
        frames=frames,
        layout=go.Layout(
            title=dict(
                text='UPF Co-authorship Network — cumulative growth (2005 \u2192 present)',
                x=0.5, xanchor='center',
            ),
            xaxis=dict(showgrid=False, zeroline=False, showticklabels=False, domain=[0, 1]),
            yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
            template='plotly_white',
            height=720,
            margin=dict(t=60, b=120),
            updatemenus=[dict(
                type='buttons', showactive=False,
                x=0.02, y=0.98, xanchor='left', yanchor='top',
                buttons=[
                    dict(label='\u25b6 Play', method='animate',
                         args=[None, dict(
                             frame=dict(duration=700, redraw=True),
                             fromcurrent=True, transition=dict(duration=0))]),
                    dict(label='\u23f8 Pause', method='animate',
                         args=[[None], dict(
                             frame=dict(duration=0, redraw=False), mode='immediate')]),
                ],
            )],
            sliders=[dict(
                active=len(anim_years) - 1,
                pad=dict(b=10, t=10),
                len=0.9, x=0.05, y=0,
                currentvalue=dict(
                    prefix='Year: ',
                    visible=True,
                    xanchor='center',
                    font=dict(size=14, color='#333'),
                ),
                transition=dict(duration=200),
                steps=[
                    dict(args=[[str(y)],
                               dict(frame=dict(duration=700, redraw=True), mode='immediate')],
                         label=str(y), method='animate')
                    for y in anim_years
                ],
            )],
        ),
    )
    fig_anim_network.show()
    print(f'Animation: {len(anim_years)} frames, {len(ey)} edge-year rows')
else:
    fig_anim_network = go.Figure()
    print('Skipped: no edge-year data')

**Reading these charts:**

**§11a — Publications per year:**  
Each bar shows how many UPF papers were published that calendar year. The dashed line is a linear regression trend — its slope quantifies the average annual growth rate of the field.

**§11b — Top 8 countries over time:**  
Lines track each country's annual paper output. Brazil's steep rise reflects the impact of Monteiro's NOVA framework, first published around 2010; US and UK growth accelerated after 2015 as the concept gained wider international traction.

**§11c — Cumulative network growth:**  
Blue shows the running total of unique researchers who have published at least one UPF paper. Red (right axis) shows unique author-pair collaborations. Both curves accelerating together means the field is not just growing in size but also becoming more densely connected.

**§11d — Animated network:**  
The same co-authorship network as §6b, but built up cumulatively year by year. Use the slider or ▶ Play button to watch the network emerge. Node colour is community membership (fixed from the full network layout). Nodes appear the first year their author published a UPF paper with at least one other author in the top-500 set; edges appear the first year the two authors co-authored.

## 7. Save enriched centrality table

In [13]:
out_path = os.path.join(OUTPUT_DIR, 'author_centrality.csv')
cols = ['author_id', 'name', 'institution', 'country', 'community',
        'papers', 'degree', 'degree_centrality', 'betweenness', 'pagerank', 'clustering']
centrality_df[cols].sort_values('betweenness', ascending=False).to_csv(out_path, index=False)
print(f'Saved enriched centrality table → {out_path}')
centrality_df[cols].head(10)

Saved enriched centrality table → ../output/author_centrality.csv


,author_id,name,institution,country,community,papers,degree,degree_centrality,betweenness,pagerank,clustering
0,https://openalex.org/A5039598820,Neha Khandpur,Universidade de São Paulo,BR,5,96,116,0.084119,0.142175,0.007424,0.009748
1,https://openalex.org/A5042007312,Carlos Augusto Monteiro,Universidade de São Paulo,BR,0,156,139,0.100798,0.129257,0.010008,0.012995
2,https://openalex.org/A5059937532,Renata Bertazzi Levy,Universidade de São Paulo,BR,0,143,165,0.119652,0.107005,0.010824,0.012151
3,https://openalex.org/A5009644208,Camila Aparecida Borges,Universidade de São Paulo,BR,0,15,14,0.010152,0.101152,0.000678,0.017414
4,https://openalex.org/A5071183177,Fernanda Rauber,Universidade de São Paulo,BR,0,93,138,0.100073,0.098458,0.007219,0.015323
5,https://openalex.org/A5064834293,Eurídice Martínez Steele,Universidade de São Paulo,BR,0,133,120,0.087020,0.096528,0.008602,0.009033
6,https://openalex.org/A5083070262,Fernanda Helena Marrocos Leite,Universidade de São Paulo,BR,0,17,38,0.027556,0.094776,0.001636,0.013123
7,https://openalex.org/A5048645214,Daniela Silva Canella,Universidade de São Paulo,BR,0,55,46,0.033358,0.080648,0.003363,0.010542
8,https://openalex.org/A5046488279,Nassib Bezerra Bueno,Universidade Federal de Alagoas,BR,10,29,31,0.022480,0.068789,0.002830,0.023066
9,https://openalex.org/A5100735336,Mengxi Du,Tufts University,US,5,39,62,0.044960,0.062690,0.003297,0.020307


## 9. Export interactive HTML dashboard

Produces a single self-contained HTML file (`output/upf_dashboard.html`) that
can be uploaded to any web server and opened in a browser — no Python or Jupyter
required.  Plotly.js is loaded from CDN; all interactivity (zoom, pan, hover,
"Papers / Citations" toggle buttons) is preserved.

In [ ]:
dashboard_figures = [
    ('Top-N Institutions',                fig_inst),
    ('Top-N Authors',                     fig_auth),
    ('Degree Distribution',               fig_deg),
    ('Co-authorship Network',             fig_network),
    ('Degree vs Betweenness Centrality',  fig_scatter),
    ('Country Collaboration Heatmap',     fig_heatmap),
    ('Author Citation Impact',            fig_cit),
    ('Citations by Community',            fig_comm_cit),
    ('Country Output vs Impact',          fig_cty),
    *([('Top Funders', fig_funders),
       ('Funding Coverage by Country', fig_fund_cty)]
      if not funders_df.empty else []),
    *([('Departments (experimental)', fig_dept)]
      if not dept_df.empty else []),
    ('Publications per Year',             fig_year),
    ('Country Trends Over Time',           fig_country_trend),
    ('Network Growth Over Time',           fig_network_growth),
    ('Network Animation',                  fig_anim_network),
]

# ── Section explanations ──────────────────────────────────────────────────────
EXPLANATIONS = {
    'Top-N Institutions': (
        'Ranks institutions by the number of UPF papers in the dataset. '
        'Toggle between <b>Papers</b> and <b>Citations</b> to distinguish volume from impact. '
        'Hover over any bar for full institution name and both metrics.'
    ),
    'Top-N Authors': (
        'Ranks individual authors by paper count or total citations. '
        'An author appearing high on papers but not citations may be a prolific contributor '
        'whose work is less internationally cited; high on citations but not papers suggests '
        'a smaller body of highly influential work.'
    ),
    'Degree Distribution': (
        'Shows how many co-authors each researcher has (left panel, linear scale) and the same '
        'distribution on log–log axes (right panel). A straight line on log–log suggests a '
        'power-law: a small number of highly connected hubs dominate the network. '
        'A steep slope means collaboration is concentrated in a few key individuals.'
    ),
    'Co-authorship Network': (
        'Each node is an author; edges connect researchers who have co-authored at least one paper. '
        '<b>Colour</b> = Louvain community (research cluster); <b>node size</b> = degree (number of direct co-authors). '
        'Click a community in the legend to isolate it. Nodes positioned close together share many collaborators. '
        'Zoom and pan to explore sub-clusters.'
    ),
    'Degree vs Betweenness Centrality': (
        '<b>X axis — Degree:</b> how many distinct co-authors an author has. High degree = a well-connected hub. '
        '<b>Y axis — Betweenness centrality:</b> how often an author sits on the shortest path between two others — '
        'i.e. how much they broker ideas across the field. '
        '<b>Point size</b> is proportional to papers published. '
        'Authors in the <b>top right</b> are both highly connected and key bridges between communities. '
        'Authors with <b>high betweenness but low degree</b> are quiet brokers: '
        'few direct collaborators, but they link groups that would otherwise be isolated.'
    ),
    'Country Collaboration Heatmap': (
        'Counts co-authored papers between every pair of countries (top-20 pairs shown). '
        'Darker cells indicate stronger bilateral collaboration. '
        'Hover for the exact count. Diagonal is empty (same-country pairs excluded).'
    ),
    'Author Citation Impact': (
        'Scatter plot of each author\'s total papers (x) against total citations (y), both on log scales. '
        '<b>Top-right</b> = prolific and highly cited. '
        '<b>Top-left</b> = few papers, each very highly cited (often review authors or field founders). '
        '<b>Bottom-right</b> = many papers, lower average impact. '
        'Point size ∝ degree (co-author count); colour = community.'
    ),
    'Citations by Community': (
        'Total citation count per research community (top 15). '
        'Hover for total papers and citations-per-paper within that community. '
        'A community with high citations but fewer papers is producing high-impact selective research; '
        'one with many papers but lower citations may be an emerging or more specialised sub-field.'
    ),
    'Country Output vs Impact': (
        'Left panel: paper count by country (volume). '
        'Right panel: citations per paper (quality proxy). '
        'Countries rank very differently on the two metrics — high volume with moderate efficiency '
        'often reflects a large national research programme; low volume with high efficiency suggests '
        'a focused, internationally visible community.'
    ),
    'Top Funders': (
        'Ranks funders by the number of UPF papers that acknowledge their support. '
        'Hover for total citations and citations-per-paper. '
        'Note: OpenAlex funding coverage is incomplete — papers without acknowledged funding '
        'are not represented here, so counts are lower bounds.'
    ),
    'Funding Coverage by Country': (
        'Left panel: percentage of a country\'s papers that record at least one funder. '
        'Right panel: funded (teal) vs no-funding-record (grey) paper counts stacked. '
        'A large grey bar can mean genuine self-funding or incomplete publisher metadata.'
    ),
    'Departments (experimental)': (
        '<strong>⚠ Experimental — partial coverage.</strong> '
        'Department names are extracted from author affiliation strings as submitted to the publisher. '
        'Only papers where a recognisable department keyword (Department, School, Centre, etc.) was found '
        'are included. The same department may appear under slightly different names across papers. '
        'Use these rankings as indicative only.'
    ),
    'Publications per Year': (
        'Each bar shows how many UPF-related papers were published that calendar year. '
        'The dashed red line is a linear regression trend — its slope quantifies average annual growth. '
        'A steepening curve would indicate accelerating interest; a flattening curve, saturation.'
    ),
    'Country Trends Over Time': (
        'Lines track each of the top 8 countries\' annual paper output. '
        'Brazil\'s steep rise from ~2010 reflects the international uptake of Monteiro\'s NOVA classification. '
        'Crossing lines reveal shifts in leadership — e.g. when the US or UK begin to match Brazilian output.'
    ),
    'Network Animation': (
        'The co-authorship network built up year by year. '
        'Use the slider at the bottom or the \u25b6 Play button to animate. '
        'Each node is an author (coloured by community), each edge a collaboration. '
        'Nodes and edges appear the first year they enter the cumulative dataset. '
        'The fixed layout means spatial position is stable across frames '
        '\u2014 clusters that form early stay together as the network grows.'
    ),
    'Network Growth Over Time': (
        'Blue (left axis): running total of unique researchers who have published at least one UPF paper. '
        'Red dashed (right axis): unique author-pair collaborations ever recorded. '
        'Both curves accelerating together means the field is not just growing in size but also becoming '
        'more densely connected — new entrants tend to co-author with existing members rather than working alone.'
    ),
}

# ── Generate narrative report from live data ──────────────────────────────────
import json as _json

_country  = pd.read_csv(os.path.join(OUTPUT_DIR, 'papers_by_country.csv'))
_country  = _country[_country['country'].notna() & (_country['country'] != '')]
_inst     = pd.read_csv(os.path.join(OUTPUT_DIR, 'papers_by_institution.csv'))
_inst     = _inst[_inst['institution'].notna() & (_inst['institution'] != '')]
_auth     = pd.read_csv(os.path.join(OUTPUT_DIR, 'papers_by_author.csv'))
_auth     = _auth[_auth['author_name'].notna() & (_auth['author_name'] != '')]

_total    = int(_country['papers'].sum())
_top_c    = _country.iloc[0]
_top_i    = _inst.iloc[0]
_top_a    = _auth.iloc[0]
_top_a2   = _auth.iloc[1]
_n_countries = len(_country)
_n_inst   = len(_inst)
_n_authors = len(_auth)
_top5_pct = round(100 * _inst.head(5)['papers'].sum() / _total, 1)
_top_c_pct = round(100 * int(_top_c['papers']) / _total, 1)
_top_i_pct = round(100 * int(_top_i['papers']) / _total, 1)

# Network stats
_n_nodes  = G_lcc.number_of_nodes()
_n_edges  = G_lcc.number_of_edges()
_n_comm   = len(communities)
_top_betw = centrality_df.sort_values('betweenness', ascending=False).iloc[0]

# Funding
_funder_note = ''
if not funders_df.empty:
    _top_f = funders_df.iloc[0]
    _funder_note = (f'The leading funder is <b>{_top_f["funder_name"]}</b> '
                    f'({int(_top_f["papers"]):,} papers), reflecting the strong '
                    f'Brazilian public research infrastructure behind this field.')

REPORT_HTML = f"""
<div class="report">
  <h2>About this analysis</h2>
  <p>
    Ultra-processed foods (UPF) are defined under the <b>NOVA classification</b>
    system developed by Carlos Augusto Monteiro and colleagues at the University
    of São Paulo. NOVA categorises foods not by nutrient content but by the extent
    and purpose of industrial processing, with Group 4 (ultra-processed) covering
    products such as soft drinks, packaged snacks, reconstituted meat products, and
    instant noodles. Since the concept was introduced in the early 2010s, the
    scientific literature on UPF and health has grown rapidly across nutrition
    epidemiology, public health policy, and food systems research.
  </p>
  <p>
    This dashboard summarises <b>{_total:,} papers</b> retrieved from
    <a href="https://openalex.org" target="_blank">OpenAlex</a> using the search
    terms <em>ultra-processed food(s)</em>, <em>ultraprocessed food(s)</em>,
    <em>ultra-processed diet</em>, and <em>NOVA food classification</em>
    in titles and abstracts. The analysis covers
    <b>{_n_authors:,} unique authors</b> across
    <b>{_n_inst:,} institutions</b> in
    <b>{_n_countries} countries</b>.
  </p>

  <h2>Key findings</h2>
  <ul>
    <li>
      <b>{_top_c["country"]} leads by paper count</b> with
      {int(_top_c["papers"]):,} papers ({_top_c_pct}% of the total), almost
      entirely driven by the University of São Paulo group.
      The US and UK are second and third.
    </li>
    <li>
      <b>{_top_i["institution"]}</b> is the single most productive institution,
      accounting for {_top_i_pct}% of all papers and
      {int(_top_i["citations"]):,} citations.
      The top 5 institutions together account for {_top5_pct}% of the literature,
      indicating a highly concentrated field.
    </li>
    <li>
      <b>{_top_a["author_name"]}</b> is the most prolific author
      ({int(_top_a["papers"])} papers, {int(_top_a["citations"]):,} citations),
      followed by <b>{_top_a2["author_name"]}</b>
      ({int(_top_a2["papers"])} papers, {int(_top_a2["citations"]):,} citations).
      Both are at the University of São Paulo.
    </li>
    <li>
      The co-authorship network (largest component) contains
      <b>{_n_nodes:,} authors</b> and <b>{_n_edges:,} collaboration links</b>,
      organised into <b>{_n_comm} research communities</b> by the Louvain algorithm.
      The most central broker — the researcher who most connects otherwise
      separate groups — is <b>{_top_betw["name"]}</b>
      ({_top_betw["institution"]}).
    </li>
    {"<li>" + _funder_note + "</li>" if _funder_note else ""}
  </ul>

  <h2>Understanding the metrics</h2>
  <dl>
    <dt><b>Citations per paper</b></dt>
    <dd>Total citations divided by paper count. A quality proxy: a country or
    institution with fewer but highly cited papers is producing work that other
    researchers rely on heavily.</dd>

    <dt><b>Co-authorship network</b></dt>
    <dd>A map of who publishes with whom. Each researcher is a node; two nodes
    are connected if they have co-authored at least one paper. The network
    reveals research communities, key collaborators, and isolated groups.</dd>

    <dt><b>Degree</b></dt>
    <dd>The number of direct co-authors an author has. A high degree means
    someone works with many different collaborators — they are a hub in the
    network.</dd>

    <dt><b>Betweenness centrality</b></dt>
    <dd>How often a researcher sits on the shortest path between two other
    researchers. A high value means they are a <em>broker</em> — they connect
    groups that would otherwise have no direct link. Even an author with a
    modest number of co-authors can have high betweenness if they bridge
    distinct communities (e.g. linking a Brazilian clinical group to a
    European epidemiology group).</dd>

    <dt><b>Louvain community</b></dt>
    <dd>A research cluster detected automatically by finding groups of authors
    who collaborate more with each other than with the rest of the network.
    Communities are colour-coded in the network graph and the scatter plot.
    Community labels show the dominant country and the most central author
    within that cluster.</dd>

    <dt><b>PageRank</b></dt>
    <dd>Borrowed from Google's web-ranking algorithm. An author's PageRank
    is high if they are connected to other highly connected authors — it
    captures prestige in the network, not just raw connectivity.</dd>
  </dl>

  <h2>Caveats</h2>
  <ul>
    <li>OpenAlex coverage is strong for journal articles but variable for
    conference papers, theses, and preprints.</li>
    <li>Author disambiguation is imperfect: the same person may appear under
    slightly different name variants.</li>
    <li>Funding metadata is incomplete — many papers carry no funder record.</li>
    <li>Department names (experimental section) are extracted heuristically
    and cover only a subset of papers.</li>
  </ul>
</div>
"""

# ── Embed search data ─────────────────────────────────────────────────────────
search_authors = (
    centrality_df[['name', 'institution', 'country', 'papers', 'citations',
                   'degree', 'betweenness', 'community_label']]
    .fillna('')
    .rename(columns={'name': 'Author', 'institution': 'Institution',
                     'country': 'Country', 'papers': 'Papers',
                     'citations': 'Citations', 'degree': 'Degree',
                     'betweenness': 'Betweenness', 'community_label': 'Community'})
    .sort_values('Papers', ascending=False)
    .head(2000)
    .to_dict(orient='records')
)
for r in search_authors:
    r['Betweenness'] = round(float(r['Betweenness']), 4)

search_inst = (
    _inst[['institution', 'country', 'papers', 'citations']]
    .fillna('')
    .rename(columns={'institution': 'Institution', 'country': 'Country',
                     'papers': 'Papers', 'citations': 'Citations'})
    .sort_values('Papers', ascending=False)
    .head(1000)
    .to_dict(orient='records')
)
search_dept = []
if not dept_df.empty:
    search_dept = (
        dept_df.fillna('')
        .rename(columns={'department': 'Department', 'institution': 'Institution',
                         'country': 'Country', 'papers': 'Papers', 'citations': 'Citations'})
        .sort_values('Papers', ascending=False)
        .head(500)
        .to_dict(orient='records')
    )

authors_json = _json.dumps(search_authors)
inst_json    = _json.dumps(search_inst)
dept_json    = _json.dumps(search_dept)

# ── CSS ───────────────────────────────────────────────────────────────────────
CSS = """
body  { font-family: Arial, Helvetica, sans-serif; max-width: 1400px;
        margin: 0 auto; padding: 24px; background: #f4f6f9; color: #333; }
h1   { font-size: 1.6rem; border-bottom: 3px solid #2c7bb6; padding-bottom: 8px; }
h2   { font-size: 1.15rem; color: #2c7bb6; margin: 40px 0 4px; }
h3   { font-size: 1rem; color: #444; margin: 20px 0 4px; }
.report { background:#fff; border-radius:8px; padding:24px 28px;
          margin-bottom:32px; box-shadow:0 1px 5px rgba(0,0,0,.1); }
.report h2 { margin-top:20px; }
.report dl { display:grid; grid-template-columns:160px 1fr; gap:6px 16px; }
.report dt { font-weight:bold; padding-top:4px; }
.report dd { margin:0; padding-top:4px; border-bottom:1px solid #eee; }
.report ul { padding-left:20px; }
.report li { margin-bottom:6px; }
.explanation { font-size:.92rem; color:#555; margin:0 0 10px;
               background:#eef4fb; border-left:3px solid #2c7bb6;
               padding:8px 12px; border-radius:0 4px 4px 0; }
.chart  { background:#fff; border-radius:8px; padding:12px;
          margin-bottom:32px; box-shadow:0 1px 5px rgba(0,0,0,.1); }
.experimental { border-left-color:#9b59b6 !important; background:#f5eefb !important; }
/* Search */
#search-section { background:#fff; border-radius:8px; padding:20px;
                  margin-bottom:32px; box-shadow:0 1px 5px rgba(0,0,0,.1); }
#search-section h2 { margin-top:0; }
.search-bar { display:flex; gap:10px; flex-wrap:wrap; margin-bottom:14px; }
#search-input { flex:1; min-width:200px; padding:8px 12px; font-size:.95rem;
                border:1px solid #ccc; border-radius:4px; }
.tab-btn { padding:7px 18px; border:none; border-radius:4px; cursor:pointer;
           font-size:.9rem; background:#e0e8f0; color:#333; }
.tab-btn.active { background:#2c7bb6; color:#fff; }
.tab-btn.dept   { background:#e8dff5; }
.tab-btn.dept.active { background:#9b59b6; }
#search-results { overflow-x:auto; }
#search-results table { border-collapse:collapse; width:100%; font-size:.88rem; }
#search-results th { background:#2c7bb6; color:#fff; padding:7px 10px;
                     text-align:left; white-space:nowrap; }
.dept-header th { background:#9b59b6 !important; }
#search-results td { padding:6px 10px; border-bottom:1px solid #eee; }
#search-results tr:hover td { background:#f0f6ff; }
#result-count { font-size:.85rem; color:#888; margin-bottom:6px; }
footer { color:#999; font-size:.8rem; margin-top:40px;
         border-top:1px solid #ddd; padding-top:8px; }
"""

SEARCH_JS = f"""
const AUTHORS = {authors_json};
const INSTS   = {inst_json};
const DEPTS   = {dept_json};
let   mode    = 'authors';

const input   = document.getElementById('search-input');
const tbody   = document.getElementById('result-body');
const thead   = document.getElementById('result-head');
const counter = document.getElementById('result-count');
const btnA    = document.getElementById('btn-authors');
const btnI    = document.getElementById('btn-inst');
const btnD    = document.getElementById('btn-dept');

function setMode(m) {{
  mode = m;
  btnA.classList.toggle('active', m === 'authors');
  btnI.classList.toggle('active', m === 'inst');
  btnD.classList.toggle('active', m === 'dept');
  const placeholders = {{
    authors: 'Search by author name, institution or country…',
    inst:    'Search by institution name or country…',
    dept:    'Search by department, institution or country…',
  }};
  input.placeholder = placeholders[m];
  render();
}}

function render() {{
  const q   = input.value.trim().toLowerCase();
  const src = mode === 'authors' ? AUTHORS : mode === 'inst' ? INSTS : DEPTS;
  const keys = {{
    authors: ['Author','Institution','Country'],
    inst:    ['Institution','Country'],
    dept:    ['Department','Institution','Country'],
  }}[mode];
  const rows = q ? src.filter(r => keys.some(k => String(r[k]).toLowerCase().includes(q)))
                 : src.slice(0, 50);

  counter.textContent = q
    ? `${{rows.length}} result${{rows.length !== 1 ? 's' : ''}} for "${{input.value}}"`
    : `Showing top 50 of ${{src.length.toLocaleString()}} — type to filter`;

  thead.className = mode === 'dept' ? 'dept-header' : '';

  if (mode === 'authors') {{
    thead.innerHTML = '<tr><th>#</th><th>Author</th><th>Institution</th><th>Country</th><th>Papers</th><th>Citations</th><th>Degree</th><th>Betweenness</th><th>Community</th></tr>';
    tbody.innerHTML = rows.slice(0,200).map((r,i) => `<tr>
      <td>${{i+1}}</td><td><b>${{r.Author}}</b></td><td>${{r.Institution}}</td>
      <td>${{r.Country}}</td><td>${{r.Papers}}</td><td>${{r.Citations.toLocaleString()}}</td>
      <td>${{r.Degree}}</td><td>${{r.Betweenness}}</td><td>${{r.Community}}</td></tr>`).join('');
  }} else if (mode === 'inst') {{
    thead.innerHTML = '<tr><th>#</th><th>Institution</th><th>Country</th><th>Papers</th><th>Citations</th></tr>';
    tbody.innerHTML = rows.slice(0,200).map((r,i) => `<tr>
      <td>${{i+1}}</td><td><b>${{r.Institution}}</b></td><td>${{r.Country}}</td>
      <td>${{r.Papers}}</td><td>${{r.Citations.toLocaleString()}}</td></tr>`).join('');
  }} else {{
    thead.innerHTML = '<tr><th>#</th><th>Department</th><th>Institution</th><th>Country</th><th>Papers</th><th>Citations</th></tr>';
    tbody.innerHTML = rows.slice(0,200).map((r,i) => `<tr>
      <td>${{i+1}}</td><td><b>${{r.Department}}</b></td><td>${{r.Institution}}</td>
      <td>${{r.Country}}</td><td>${{r.Papers}}</td><td>${{r.Citations.toLocaleString()}}</td></tr>`).join('');
  }}
}}

input.addEventListener('input', render);
render();
"""

SEARCH_WIDGET = """
<div id="search-section">
  <h2>&#128269; Search Authors, Institutions &amp; Departments</h2>
  <div class="search-bar">
    <button class="tab-btn active" id="btn-authors" onclick="setMode('authors')">Authors</button>
    <button class="tab-btn"        id="btn-inst"    onclick="setMode('inst')">Institutions</button>
    <button class="tab-btn dept"   id="btn-dept"    onclick="setMode('dept')">Departments &#9888;</button>
    <input id="search-input" type="text"
           placeholder="Search by author name, institution or country…" />
  </div>
  <div id="result-count"></div>
  <div id="search-results">
    <table><thead id="result-head"></thead><tbody id="result-body"></tbody></table>
  </div>
</div>
"""

# ── Build HTML ────────────────────────────────────────────────────────────────
generated_at = pd.Timestamp.now().strftime('%Y-%m-%d %H:%M')

header = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>UPF Bibliometrics — Interactive Dashboard</title>
<style>{CSS}</style>
</head>
<body>
<h1>UPF Bibliometrics — Interactive Dashboard</h1>
<p>Generated: {generated_at} &nbsp;|&nbsp;
   Source: <a href="https://openalex.org" target="_blank">OpenAlex</a></p>
{REPORT_HTML}
{SEARCH_WIDGET}
"""

footer = f"""
<script>{SEARCH_JS}</script>
<footer>Generated with <a href="https://plotly.com/python/" target="_blank">Plotly</a>
and the UPF bibliometrics pipeline.</footer>
</body></html>"""

parts = [header]
first = True
for section_title, fig in dashboard_figures:
    expl = EXPLANATIONS.get(section_title, '')
    is_exp = 'experimental' in section_title.lower()
    expl_cls = 'explanation experimental' if is_exp else 'explanation'
    expl_html = f'<p class="{expl_cls}">{expl}</p>' if expl else ''
    chart_html = fig.to_html(
        full_html=False,
        include_plotlyjs='cdn' if first else False,
        config={'displayModeBar': True, 'scrollZoom': False},
    )
    parts.append(
        f'<h2>{section_title}</h2>\n{expl_html}\n<div class="chart">\n{chart_html}\n</div>'
    )
    first = False

parts.append(footer)

html_path = os.path.join(OUTPUT_DIR, 'upf_dashboard.html')
with open(html_path, 'w', encoding='utf-8') as fh:
    fh.write('\n'.join(parts))

size_kb = os.path.getsize(html_path) / 1024
print(f'Saved → {html_path}  ({size_kb:.0f} KB)')


## §12 Study Type Analysis

Classifies each paper as **RCT**, **Observational** (cohort / cross-sectional /
case-control), **Systematic Review / Meta-analysis**, **Clinical Trial**, **Review**,
or **Other**.

Classification priority:
1. PubMed MeSH publication-type tags (most reliable; covers ~65–70 % of the corpus)
2. OpenAlex work type (`review`)
3. Title-keyword heuristics

Requires `papers_by_study_type.csv`, `papers_by_study_type_year.csv`, and
`papers_by_author_study_type.csv` — all written by `upf_bibliometrics.py`.

In [ ]:
ST_COLORS = {
    "Observational":                     "#27ae60",
    "Systematic Review / Meta-analysis": "#8e44ad",
    "Review":                            "#9b59b6",
    "RCT":                               "#2980b9",
    "Clinical Trial":                    "#e67e22",
    "Other":                             "#95a5a6",
}
ST_SYMBOLS = {
    "Observational":                     "circle",
    "Systematic Review / Meta-analysis": "diamond",
    "Review":                            "square",
    "RCT":                               "triangle-up",
    "Clinical Trial":                    "cross",
    "Other":                             "circle-open",
}

_st_csv      = os.path.join('..', 'output', 'papers_by_study_type.csv')
_st_year_csv = os.path.join('..', 'output', 'papers_by_study_type_year.csv')
_st_auth_csv = os.path.join('..', 'output', 'papers_by_author_study_type.csv')

if not os.path.exists(_st_csv):
    print("⚠  Study type data not found — re-run upf_bibliometrics.py first")
    st_df = st_year_df = st_auth_df = None
else:
    st_df      = pd.read_csv(_st_csv)
    st_year_df = pd.read_csv(_st_year_csv)
    st_auth_df = pd.read_csv(_st_auth_csv)
    total_st   = int(st_df['papers'].sum())
    print(f"Study type data loaded: {total_st:,} papers across {len(st_df)} categories")
    display(st_df.sort_values('papers', ascending=False))

In [ ]:
if st_df is not None:
    st_plot = st_df.sort_values('papers', ascending=False).reset_index(drop=True)
    colors  = [ST_COLORS.get(t, '#aaa') for t in st_plot['study_type']]

    fig_st = make_subplots(
        rows=1, cols=2,
        specs=[[{'type': 'domain'}, {'type': 'xy'}]],
        subplot_titles=('Proportion', 'Paper count'),
    )

    # Donut
    fig_st.add_trace(go.Pie(
        labels=st_plot['study_type'],
        values=st_plot['papers'],
        hole=0.42,
        marker=dict(colors=colors, line=dict(color='white', width=2)),
        textinfo='label+percent',
        textfont=dict(size=11),
        hovertemplate='<b>%{label}</b><br>Papers: %{value:,}<br>%{percent}<extra></extra>',
        showlegend=False,
    ), row=1, col=1)

    # Horizontal bar
    fig_st.add_trace(go.Bar(
        y=st_plot['study_type'][::-1],
        x=st_plot['papers'][::-1],
        orientation='h',
        marker=dict(color=colors[::-1]),
        text=st_plot['papers'][::-1].apply(lambda v: f'{int(v):,}'),
        textposition='inside',
        insidetextanchor='end',
        textfont=dict(color='white', size=11),
        hovertemplate='<b>%{y}</b><br>Papers: %{x:,}<extra></extra>',
        showlegend=False,
    ), row=1, col=2)

    fig_st.update_layout(
        title='§12a  Study type distribution',
        height=360,
        xaxis=dict(title='Papers', range=[0, int(st_plot['papers'].max()) * 1.08]),
        margin=dict(t=60, b=20, l=20, r=20),
        plot_bgcolor='white', paper_bgcolor='white',
    )
    fig_st.show()

**Reading this chart:**

**Left (donut):** share of each study design in the corpus.
Observational studies (cohort, cross-sectional, case-control) typically dominate —
reflecting the epidemiological foundations of UPF research.

**Right (bar):** raw paper counts.
*Other* contains papers that could not be classified by MeSH tags or title keywords;
this fraction will be smaller once all papers are indexed in PubMed.

**Caveats:**  
- MeSH tags are applied retrospectively by NLM — recent papers and non-English work
  may not yet have them, increasing the *Other* fraction.  
- A paper with both `Cohort Studies` and `Randomized Controlled Trial` MeSH tags
  is classified as RCT (priority order: RCT > Meta-analysis > Observational …).
- When MeSH tags are absent, the title is searched for any form of *randomis/randomiz*
  (randomised, randomized, randomisation, randomized crossover, etc.) before falling
  back to keyword matching for other study types.

In [ ]:
if st_year_df is not None:
    # Restrict to 2000–2024 for readability
    _yr = st_year_df[st_year_df['year'].between(2000, 2024)].copy()
    years = sorted(_yr['year'].unique())
    type_order = st_df.sort_values('papers', ascending=False)['study_type'].tolist()

    fig_st_yr = go.Figure()
    for stype in type_order:
        sub = _yr[_yr['study_type'] == stype].set_index('year')
        vals = [int(sub.loc[y, 'papers']) if y in sub.index else 0 for y in years]
        fig_st_yr.add_trace(go.Bar(
            name=stype,
            x=years,
            y=vals,
            marker_color=ST_COLORS.get(stype, '#aaa'),
            hovertemplate=f'<b>{stype}</b><br>Year: %{{x}}<br>Papers: %{{y:,}}<extra></extra>',
        ))

    fig_st_yr.update_layout(
        title='§12b  Study types over time (2000–2024)',
        barmode='stack',
        height=400,
        xaxis=dict(title='Year'),
        yaxis=dict(title='Papers published'),
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0),
        plot_bgcolor='white', paper_bgcolor='white',
        margin=dict(t=80, b=40, l=50, r=20),
    )
    fig_st_yr.show()

**Reading this chart:**

Absolute paper counts per year, stacked by study design.

Things to look for:
- Growth of systematic reviews and meta-analyses — these typically follow a maturing
  evidence base, so their rise relative to observational work indicates the field
  is consolidating.
- Whether RCT output has kept pace with observational growth (a common critique of
  the UPF field is the relative paucity of RCTs).
- The *Other* stack will shrink as PubMed MeSH back-tagging catches up.

In [ ]:
if st_auth_df is not None:
    # Join author study-type table with author country via papers_by_author.csv
    _auth_ctr = authors_df[['author_id', 'country']].drop_duplicates(subset='author_id')
    _st_ctr   = st_auth_df.merge(_auth_ctr, on='author_id', how='left')
    _st_ctr   = _st_ctr[_st_ctr['country'].notna() & (_st_ctr['country'] != '')]

    # Top 15 countries by total attributed author-papers
    top_ctr = (
        _st_ctr.groupby('country')['papers'].sum()
        .nlargest(15).index.tolist()
    )
    _st_ctr_top = _st_ctr[_st_ctr['country'].isin(top_ctr)]

    pivot = (
        _st_ctr_top.groupby(['country', 'study_type'])['papers']
        .sum().reset_index()
        .pivot(index='country', columns='study_type', values='papers')
        .fillna(0)
    )
    # Normalise to 100 %
    pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100

    # Sort countries by total paper count
    pivot_pct = pivot_pct.loc[
        pivot.sum(axis=1).sort_values(ascending=False).index
    ]

    type_order = st_df.sort_values('papers', ascending=False)['study_type'].tolist()
    present = [t for t in type_order if t in pivot_pct.columns]

    fig_ctr = go.Figure()
    for stype in present:
        fig_ctr.add_trace(go.Bar(
            name=stype,
            x=pivot_pct.index,
            y=pivot_pct[stype].round(1),
            marker_color=ST_COLORS.get(stype, '#aaa'),
            hovertemplate=f'<b>{stype}</b><br>%{{x}}: %{{y:.1f}} %<extra></extra>',
        ))

    fig_ctr.update_layout(
        title='§12c  Study type mix by country (top 15, % of author-attributed papers)',
        barmode='stack',
        height=420,
        xaxis=dict(title='Country (ISO code)'),
        yaxis=dict(title='%', range=[0, 100]),
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0),
        plot_bgcolor='white', paper_bgcolor='white',
        margin=dict(t=80, b=40, l=50, r=20),
    )
    fig_ctr.show()

**Reading this chart:**

Each bar shows 100 % of the author-attributed papers for a country, split by study
design.

**Note on counting:** A paper with three authors from the same country contributes
three times to that country's total (once per author). This inflates countries with
many co-authors per paper but is consistent with the rest of the bibliometric
analysis.  Compare % rather than absolutes across countries.

Things to look for:
- Do high-output countries (BR, US) differ in study-type mix?
- Are there countries with a notably high RCT share — often a sign of clinical or
  interventional nutrition research infrastructure?

In [ ]:
if st_auth_df is not None:
    # Dominant study type per author (plurality rule)
    _dom_st = {}
    for _, r in st_auth_df.iterrows():
        aid = r['author_id']
        if pd.isna(aid):
            continue
        if aid not in _dom_st or int(r['papers']) > _dom_st[aid]['papers']:
            _dom_st[aid] = {'study_type': r['study_type'], 'papers': int(r['papers'])}

    # Add study_type to centrality_df (fallback: 'Other')
    _cent = centrality_df.copy()
    _cent['study_type'] = _cent['author_id'].map(
        lambda aid: _dom_st.get(aid, {}).get('study_type', 'Other')
    )

    # Spring layout (same seed = same positions as §6b)
    _pos = nx.spring_layout(G_lcc, weight='weight', seed=LAYOUT_SEED, k=0.5)

    # Cap to top PLOT_CAP nodes by degree
    PLOT_CAP_ST = 500
    if G_lcc.number_of_nodes() > PLOT_CAP_ST:
        _top_nodes = sorted(G_lcc.nodes(), key=lambda n: G_lcc.degree(n), reverse=True)[:PLOT_CAP_ST]
        _G_plot = G_lcc.subgraph(_top_nodes).copy()
    else:
        _G_plot = G_lcc

    _node_ids  = list(_G_plot.nodes())
    _node_idx  = {nid: i for i, nid in enumerate(_node_ids)}
    _cent_idx  = _cent.set_index('author_id')

    # Edge trace
    ex, ey = [], []
    for a, b in _G_plot.edges():
        ex += [_pos[a][0], _pos[b][0], None]
        ey += [_pos[a][1], _pos[b][1], None]

    fig_st_net = go.Figure()
    fig_st_net.add_trace(go.Scatter(
        x=ex, y=ey, mode='lines',
        line=dict(width=0.4, color='rgba(140,140,140,0.2)'),
        hoverinfo='none', showlegend=False,
    ))

    # One trace per study type
    type_order_net = list(ST_COLORS.keys())
    for stype in type_order_net:
        nx_, ny_, ns_, nt_ = [], [], [], []
        for nid in _node_ids:
            st_n = _dom_st.get(nid, {}).get('study_type', 'Other')
            if st_n != stype:
                continue
            try:
                row = _cent_idx.loc[nid]
                pap = int(row['papers']) if 'papers' in row else 0
                cit = int(row['citations']) if 'citations' in row else 0
                name = str(row['name']) if 'name' in row else nid
                inst = str(row['institution']) if 'institution' in row else ''
            except KeyError:
                pap, cit, name, inst = 0, 0, str(nid), ''
            deg = _G_plot.degree(nid)
            nx_.append(_pos[nid][0])
            ny_.append(_pos[nid][1])
            ns_.append(max(5, min(20, 4 + deg * 0.2)))
            nt_.append(f'<b>{name}</b><br>{inst}<br>Study type: {stype}<br>'
                       f'Papers: {pap} · Citations: {cit:,}<extra></extra>')
        if not nx_:
            continue
        fig_st_net.add_trace(go.Scatter(
            name=stype,
            x=nx_, y=ny_, mode='markers',
            marker=dict(
                symbol=ST_SYMBOLS.get(stype, 'circle'),
                size=ns_,
                color=ST_COLORS.get(stype, '#aaa'),
                opacity=0.85,
                line=dict(width=0.6, color='white'),
            ),
            hovertemplate=nt_,
            showlegend=True,
        ))

    fig_st_net.update_layout(
        title='§12d  Co-authorship network — nodes coloured & shaped by dominant study type',
        height=680,
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False, scaleanchor='x'),
        hovermode='closest',
        plot_bgcolor='#f9fafc', paper_bgcolor='#f9fafc',
        legend=dict(
            title='Study type', x=0.01, y=0.99,
            bgcolor='rgba(255,255,255,0.85)',
            bordercolor='#ddd', borderwidth=1, font=dict(size=11),
        ),
        margin=dict(t=60, b=10, l=10, r=10),
    )
    fig_st_net.show()

**Reading this chart:**

Node **colour and shape** show each author's dominant study type (the design
category with the most of their UPF papers):

| Shape | Study type |
|-------|-----------|
| ● circle | Observational |
| ◆ diamond | Systematic Review / Meta-analysis |
| ■ square | Review |
| ▲ triangle | RCT |
| ✚ cross | Clinical Trial |
| ○ open circle | Other / unclassified |

Node **size** scales with degree (number of direct co-authors visible).

Click legend entries to isolate a study type.  Look for:
- Whether RCT researchers form tight clusters or are distributed across the network.
- Whether systematic reviewers occupy bridging positions (high betweenness) — their
  multi-group citations often make them connectors between empirical sub-communities.